# Pan-protein DMS &mdash; Livesey &amp; Marsh compiled predictions

Source: Livesey &amp; Marsh, *Genome Biology* 2025, per-protein predictor tables (figshare item 28295198).

Builds `data/other_benchmarks/marsh_gen_bio_2025.parquet` from the authors' standardised
compilation: one CSV per protein, named by UniProt accession (`P60484.csv`, ...), each a
wide table of `variant` + ~110 predictor columns, several of which are the DMS assays
themselves (`DMS`, `DMS_b`, `DMS_100_WT`, `DMS_urn-mavedb-...`, ...).

Steps:

1. **melt** &mdash; per file, unpivot every column named `DMS` or `DMS_*` into
   `(DMS_assay, DMS_score)`, drop null scores, tag `uniprot_id` from the file name.
2. **parse** `variant` (`M1A`) &rarr; `ref_aa` / `aa_position` / `alt_aa`; `mutant` = the
   raw `variant` string (the files are missense-only).
3. **region** &mdash; `uniprot_id` &rarr; Ensembl gene id via the UniProt human idmapping
   file, **many-to-many, no dedup** (same source/behaviour as
   `utils/annotations/add_more_annotations.py:_build_uniprot_map`). 3 of the 36 accessions
   carry 2 Ensembl ids, so the join must not collapse them; `.unique()` afterwards.
   `P0DP23` (calmodulin) would carry 3 more (CALM1/2/3) in the current idmapping release,
   so it is pinned to `ENSG00000198668` to match the shipped file.

The final cell asserts the result matches the shipped file exactly: 601 270 rows, the same
105 `DMS_assay` values, same content. (Verified: a pandas prototype of this transform
reproduces the shipped parquet row-for-row.)

In [ ]:
import glob
import pathlib
import urllib.request
import polars as pl

REPO_ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (p / 'utils' / 'variant_filtering.py').exists())

# One CSV per UniProt accession (figshare item 28295198).
RAW_DIR = pathlib.Path('~/Downloads/28295198').expanduser()

WORK = REPO_ROOT / 'data' / '_raw' / 'marsh'
WORK.mkdir(parents=True, exist_ok=True)
OUT = REPO_ROOT / 'data' / 'other_benchmarks' / 'marsh_gen_bio_2025.parquet'

UNIPROT_IDMAP = ('https://ftp.uniprot.org/pub/databases/uniprot/current_release/'
                 'knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz')

In [ ]:
csvs = sorted(glob.glob(str(RAW_DIR / '*.csv')))
print(len(csvs), 'per-protein CSVs')
print([pathlib.Path(p).stem for p in csvs])

## 1&ndash;2. Melt DMS columns and parse `variant`

In [ ]:
OUT_COLS = ['uniprot_id', 'mutant', 'DMS_assay', 'DMS_score',
            'ref_aa', 'aa_position', 'alt_aa', 'region']

parts = []
for path in csvs:
    uid = pathlib.Path(path).stem
    df = pl.read_csv(path, infer_schema_length=50000)
    dcols = [c for c in df.columns if c == 'DMS' or c.startswith('DMS_')]
    if not dcols:
        print(f'  {uid}: no DMS column, skipped')
        continue
    parts.append(
        df.select(['variant', *dcols])
        .with_columns(pl.col(dcols).cast(pl.Float64, strict=False))  # empty cols read as str
        .unpivot(index='variant', on=dcols,
                 variable_name='DMS_assay', value_name='DMS_score')
        .drop_nulls('DMS_score')
        .with_columns(uniprot_id=pl.lit(uid))
    )

long = (
    pl.concat(parts)
    .with_columns(
        ref_aa=pl.col('variant').str.slice(0, 1),
        alt_aa=pl.col('variant').str.slice(-1, 1),
        aa_position=pl.col('variant').str.slice(1, pl.col('variant').str.len_chars() - 2)
                      .cast(pl.Int64),
        mutant=pl.col('variant'),
    )
)
print(long.shape)
long.head()

## 3. `uniprot_id` &rarr; Ensembl gene id (many-to-many, no dedup)

In [ ]:
idmap_path = WORK / 'HUMAN_9606_idmapping.dat.gz'
if not idmap_path.exists():
    print('downloading UniProt idmapping (~120 MB)')
    urllib.request.urlretrieve(UNIPROT_IDMAP, idmap_path)

idmap = (
    pl.read_csv(idmap_path, separator='\t', has_header=False,
                new_columns=['uniprot_id', 'id_type', 'value'])
    .filter(pl.col('id_type') == 'Ensembl')
    .with_columns(region=pl.col('value').str.split('.').list.first())
    .select(['uniprot_id', 'region'])
    .unique()
)

# P0DP23 (calmodulin) is encoded by 3 identical genes CALM1/2/3; the UniProt
# current_release idmapping now lists all 3 Ensembl ids. Pin it to CALM3
# (ENSG00000198668, as the shipped file does) so the join doesn't triplicate
# every calmodulin row. The other multi-region accessions (2 ids each) pass through.
idmap = idmap.filter(
    (pl.col('uniprot_id') != 'P0DP23') | (pl.col('region') == 'ENSG00000198668')
)

print(idmap.height, 'uniprot->ensembl rows;',
      idmap.filter(pl.col('uniprot_id').is_in(long['uniprot_id'].unique()))
           .group_by('uniprot_id').len().filter(pl.col('len') > 1).height,
      'of our accessions map to >1 region')

In [ ]:
marsh = (
    long.join(idmap, on='uniprot_id', how='inner')   # duplicates rows for 2-region accessions
    .select(OUT_COLS)
    .unique()
)
print(marsh.shape)
marsh.head()

## Verify against the shipped HuggingFace file

In [ ]:
import pyarrow.parquet as pq

EXPECTED_DMS_ASSAYS = ['DMS', 'DMS_0', 'DMS_0.15', 'DMS_0.625', 'DMS_100_A222V', 'DMS_100_WT', 'DMS_12_A222V', 'DMS_12_WT', 'DMS_200_A222V', 'DMS_200_WT', 'DMS_25_A222V', 'DMS_25_WT', 'DMS_5', 'DMS_AP2B1', 'DMS_BrefeldinA', 'DMS_DNMDP_100', 'DMS_DOX', 'DMS_Dopamine', 'DMS_Geldanamycin', 'DMS_MG-132', 'DMS_Melatonin', 'DMS_Menadione', 'DMS_Miconazole', 'DMS_OBFC1', 'DMS_Rapamycin', 'DMS_SCH', 'DMS_Spermidine', 'DMS_Tunicamycin', 'DMS_VAMP', 'DMS_VRT', 'DMS_WT_Nutlin', 'DMS_abundance', 'DMS_activity', 'DMS_atorvastatin', 'DMS_attenuated', 'DMS_b', 'DMS_b_fitness', 'DMS_b_refined', 'DMS_blx', 'DMS_blx_geneticin', 'DMS_expression', 'DMS_filtered_stability', 'DMS_flipped', 'DMS_full', 'DMS_g12v', 'DMS_growth', 'DMS_halflife', 'DMS_highB6', 'DMS_highqual_b', 'DMS_ibrutinib', 'DMS_le9', 'DMS_le9_geneticin', 'DMS_lowB6', 'DMS_no_statin', 'DMS_norm', 'DMS_null_Nutlin', 'DMS_null_etoposide', 'DMS_refined_highB6', 'DMS_refined_lowB6', 'DMS_regulated', 'DMS_rosuvastatin', 'DMS_sensitivity', 'DMS_trequinsin_100', 'DMS_unregulated', 'DMS_urn-mavedb-00000041-a-1', 'DMS_urn-mavedb-00000041-b-1', 'DMS_urn-mavedb-00000114-a-1', 'DMS_urn-mavedb-00000115-a-1', 'DMS_urn-mavedb-00000115-a-10', 'DMS_urn-mavedb-00000115-a-11', 'DMS_urn-mavedb-00000115-a-12', 'DMS_urn-mavedb-00000115-a-13', 'DMS_urn-mavedb-00000115-a-14', 'DMS_urn-mavedb-00000115-a-15', 'DMS_urn-mavedb-00000115-a-16', 'DMS_urn-mavedb-00000115-a-17', 'DMS_urn-mavedb-00000115-a-18', 'DMS_urn-mavedb-00000115-a-19', 'DMS_urn-mavedb-00000115-a-2', 'DMS_urn-mavedb-00000115-a-20', 'DMS_urn-mavedb-00000115-a-21', 'DMS_urn-mavedb-00000115-a-22', 'DMS_urn-mavedb-00000115-a-23', 'DMS_urn-mavedb-00000115-a-24', 'DMS_urn-mavedb-00000115-a-25', 'DMS_urn-mavedb-00000115-a-26', 'DMS_urn-mavedb-00000115-a-27', 'DMS_urn-mavedb-00000115-a-28', 'DMS_urn-mavedb-00000115-a-3', 'DMS_urn-mavedb-00000115-a-4', 'DMS_urn-mavedb-00000115-a-5', 'DMS_urn-mavedb-00000115-a-6', 'DMS_urn-mavedb-00000115-a-7', 'DMS_urn-mavedb-00000115-a-8', 'DMS_urn-mavedb-00000115-a-9', 'DMS_urn-mavedb-00000657-a-1', 'DMS_urn-mavedb-00000657-a-2', 'DMS_urn-mavedb-00000659-a-1', 'DMS_urn-mavedb-00000660-a-1', 'DMS_urn-mavedb-00000661-a-1', 'DMS_urn-mavedb-00000661-b-1', 'DMS_urn-mavedb-00000661-c-1', 'DMS_urn-mavedb-00000661-d-1', 'DMS_urn-mavedb-00001079-a-1', 'DMS_zscore_ratio']

old = pl.from_arrow(pq.read_table(OUT))

assert marsh.columns == old.columns, (marsh.columns, old.columns)
assert marsh.height == old.height == 601270, (marsh.height, old.height)
assert sorted(marsh['DMS_assay'].unique().to_list()) == EXPECTED_DMS_ASSAYS, \
    set(EXPECTED_DMS_ASSAYS).symmetric_difference(marsh['DMS_assay'].unique().to_list())

key = ['uniprot_id', 'region', 'DMS_assay', 'mutant', 'DMS_score']
a, b = marsh.sort(key), old.sort(key)
assert a.drop('DMS_score').equals(b.drop('DMS_score')), 'non-score columns differ'
assert (a['DMS_score'] - b['DMS_score']).abs().max() < 1e-9, 'DMS_score differs'
print('content matches the shipped marsh_gen_bio_2025.parquet')

In [ ]:
# Overwrite the shipped file (only when the assert above passes and you mean to).
# marsh.write_parquet(OUT)